<h1><center> <ins><b> Binomial Option Pricing Model Code </b></ins></center></h1>

Based of all the work we did in **"BOPM Derivations and Concepts"**, we will implement 3 different models. They will be as follows:

1) A model where we are given $S, K, r, q, u, d, T, n$

2) A model based of the CRR parameters where we are given $S, K, r, q, \sigma, T, n$

3) A model based of Chance parameters where we are given $S, K, r, q, \sigma, T, n, \bar{p}$

These variable represent:
- $S$ : Price of the underlying asset
- $K$ : Strike Price of option
- $r$ : Risk-free interest rate
- $q$ : Continuous dividend yield
- $\sigma$ : Volatility
- $u$ : Up factor that has a $p$ probability of increasing the value of $S$ to $uS$
- $d$ : Up factor that has a $1-p$ probability of decreasing the value of $S$ to $dS$
- $T$ : Time to expiry of option
- $n$ : Number of time steps
- $\bar{p}$ : The choice for our risk-free probability in the Chance Model

For each model, there are 2 main methods to implement them.

The first is by finding the value of the option at expiry of each possible node the underlying asset can reach after a specific number of up and down movements, then by taking 2 at a time, working backwards along the tree to finally find the initial value of the option.\
The second is by using our formulae we derived in **"BOPM Derivations and Concepts"** that skips travelling back along the tree, and instead simply uses our given parameters and a Binomial CDF.

It should be clear that the second implementation is far superior since our function would run in $O(1)$ time, whereas for the first, it would be $O(n)$. Since, our model is more accurate the greater the value of $n$, it is obvious we should use the second implementation. However, for the sake of comparison we will implement both ways for our first model and see how great the difference is.

In [1]:
# Import libraries

import numpy as np
import time
from scipy.stats import binom

<h1> <ins>Model 1</ins> </h1>

## Slow Implementation

From **"BOPM Derivations and Concepts"** (5),
$$V = e^{-r\delta t}\left(\bar{p}V^{+} + \left(1-\bar{p}\right)V^{-}\right) \tag{1}$$\
$$\text{where}\,\,\bar{p} = \frac{e^{r\delta t}-d}{u-d}$$\
where $V$ is the value of an option at time $t=x\,\,,(x = k\delta t\,\,s.t\,\,x\in [0,T])$, $V^{+}$ is the value of an option when $S$ experiences an up movement at $t=x+\delta t$, and $V^{-}$ is the value of an option when $S$ experiences a down movement at $t=x+\delta t$\.

We will implement this by following these steps:
- Starting with $S$, we will create a tree that stores all possible prices the underlying asset can become.\ We will do this by starting with an array with 1 element, our initial underlying asset price. Then for each time step, the number of possible prices the asset can become increases by 1. An easy way to capture this idea is by multiplying our previous time step array all by $u$ and then appending the last element multiplied by $d$ to get all possible current time step prices. For example, suppose our previous possible asset prices are $[u^{2}S, udS, d^{2}S]$, our current possible prices are $[u^{3}S, u^{2}dS, ud^{2}S]$ with $[d^{3}S]$, resulting in $[u^{3}S, u^{2}dS, ud^{2}S, d^{3}S]$.

- Then, we will look at the $n+1$ possible prices of the underlying asset at expiry and calculate the value of an option at expiry by considering the payoff function $\text{Payoff} = \text{max}\{S-K,0\}$ for each possible price.

- Finally, we will work backwards along our tree to find the initial value of the option at time $t=0$. To do this, we start with an array of the values of option at expiry. Then for the previous time step, we take 2 consecutive elements and use the formula (1) to find the values of option at the previous time step and repeat this until we perform it on the last 2 elements. We repeat this until we have our inital value of the option.

Now let us implement this.

In [2]:
def BOPM_Slow(S, K, r, q, u, d, T, n, Option_Type):

    # Check to see if we have a valid option type
    if Option_Type not in ('call','put'):
        print('Invalid option type, please enter either \'call\' or \'put\'')
        return
    
    # Initialise arrays for stock prices (S) and option values (V)
    SP = [0] * (n + 1)
    VP = [0] * (n + 1)
    
    # Calculate time step and discount factor
    dt = T/n
    df = np.exp(-r * dt)
    
    # Calculate value for p
    p = (np.exp((r-q) * dt) - d) / (u - d)

    # Set initial asset price
    SP[0] = S

    # Simulate the stock prices at each step
    for n in range(1, n + 1):
        for j in range(0, n):
            SP[j] = u * SP[j]
        SP[n] = (d/u) * SP[n-1]

    # Calculate option values at expiration depending if call or put
    if Option_Type == 'call':
        for j in range(n + 1):
            VP[j] = max(SP[j] - K,0)
    else:
        for j in range(n + 1):
            VP[j] = max(K - SP[j],0)

    # Calculate option values at previous time steps using backward recursion
    for n in range(n, 0, -1):
        for j in range(n):
            VP[j] = (p * VP[j] + (1 - p) * VP[j + 1]) * df
    
    # Return the option price at the root
    print("The value of our {0} option is {1:0.2f}".format(Option_Type,VP[0]))

    return VP[0]

## Fast implementation

From **"BOPM Derivations and Concepts"** (13)-(17) for a **call option**,
$$V = SB\left(a;n,\hat{p}\right) - Ke^{-rT}B\left(a;n,\bar{p}\right)$$\
$$\text{where}\,\,B\left(x;n,p\right) = \sum_{j=x}^{n}\binom{n}{j}p^{j}\left(1-p\right)^{n-j} = 1 - \mathbb{P}\left(X \le x-1\right)\,\,\text{where}\,\,X \thicksim \text{Bin}\left(n,p\right)$$\
$$\text{where}\,\,a \in \mathbb{N}\,\,\text{s.t}\,\,a > \frac{\ln{\left(\frac{K}{Sd^{n}}\right)}}{\ln{\left(\frac{u}{d}\right)}}$$\
$$\text{and}\,\,\bar{p} = \frac{e^{r\delta t}-d}{u-d}$$\
$$\text{and}\,\,\hat{p} = \frac{u}{e^{r\delta t}}\bar{p} = \frac{u\left(e^{r\delta t} - d\right)}{e^{r\delta t}\left(u-d\right)}$$

And for a **put option**,
$$V = Ke^{-rT}\tilde{B}\left(a;n,\bar{p}\right) - S\tilde{B}\left(a;n,\hat{p}\right)$$\
$$\text{where}\,\,\tilde{B}\left(x;n,p\right) = \sum_{j=0}^{x}\binom{n}{j}p^{j}\left(1-p\right)^{n-j} = \mathbb{P}\left(X \le x\right)\,\,\text{where}\,\,X \thicksim \text{Bin}\left(n,p\right)$$\
$$\text{where}\,\,a \in \mathbb{N}\,\,\text{s.t}\,\,a < \frac{\ln{\left(\frac{K}{Sd^{n}}\right)}}{\ln{\left(\frac{u}{d}\right)}}$$\
$$\text{and}\,\,\bar{p} = \frac{e^{r\delta t}-d}{u-d}$$\
$$\text{and}\,\,\hat{p} = \frac{u}{e^{r\delta t}}\bar{p} = \frac{u\left(e^{r\delta t} - d\right)}{e^{r\delta t}\left(u-d\right)}$$

In order to implement this, we will use SciPy's binomial cdf function to help with our $B$ function. The other thing of note is that if our $d < 0$, if we look inside our log function when calculating $a$, we will be computing $d^{n}$. For large $n$, we will have float division by zero errors.

So we will rewrite the definition of $a$ in the **call option** equation slightly to
$$a \in \mathbb{N}\,\,\text{s.t}\,\,a > \frac{\ln{\left(\frac{K}{S}\right)} + \ln{\left(d^{-n}\right)}}{\ln{\left(\frac{u}{d}\right)}}$$\
$$\implies a \in \mathbb{N}\,\,\text{s.t}\,\,a > \frac{\ln{\left(\frac{K}{S}\right)} - n \times \ln{\left(d\right)}}{\ln{\left(\frac{u}{d}\right)}}$$

And similarly rewrite the definition of $a$ in the **put option** equation to
$$a \in \mathbb{N}\,\,\text{s.t}\,\,a < \frac{\ln{\left(\frac{K}{S}\right)} - n \times \ln{\left(d\right)}}{\ln{\left(\frac{u}{d}\right)}}$$

Now let us implement this.

In [3]:
def BOPM_Fast(S, K, r, q, u, d, T, n, Option_Type):

    # Check to see if we have a valid option type
    if Option_Type not in ('call','put'):
        print('Invalid option type, please enter either \'call\' or \'put\'')
        return
    
    # Calculate time step and discount factor
    dt = T/n
    df = np.exp(-r * dt)
    
    # Calculate our value for 'a'
    ac = np.ceil((np.log(K/S) - (n * np.log(d))) / (np.log(u/d)))
    ap = np.floor((np.log(K/S) - (n * np.log(d))) / (np.log(u/d))) + 1
    
    # Calculate our 2 risk-neutral probabilities
    p1 = (np.exp((r-q) * dt) - d) / (u - d)
    p2 = u * df * p1
    
    # Create our B lambda function
    B = lambda x, n, p: binom(n, p).cdf(x-1)
    
    if Option_Type == 'call':
        # Calculate the inital value of the option if it is a call
        V = (S * (1 - B(ac, n, p2))) - (K * np.exp(-r * T) * (1 - B(ac, n, p1)))
        
    else:
        # Calculate the inital value of the option if it is a put
        V = (K * np.exp(-r * T) * B(ap, n, p1)) - (S * B(ap, n, p2))
    
    # Return the option price at the root
    #print("The value of our {0} option is {1:0.2f}".format(Option_Type,V))
    
    return V

## Comparison of implementation

Now we will compare our implementation methods by running both functions for a small $n$ and then again for a large $n$. It is hard to see the difference for small $n$ and the effeciency difference will be clear for larger values. 

In [4]:
# Parameters

S = 100
K = 100
r = 0.1
q = 0
u = 1.2
d = 0.8
T = 5

In [5]:
# Let us use a small n first and consider a call option
n = 10
Option_Type = 'call'

# Note start time
start = time.time()
# Run function
BOPM_Slow(S, K, r, q, u, d, T, n, Option_Type)
# Note end time and calculate difference
end = time.time()
print(f'The time elapsed whilst running the Slow Method with {n} time steps is {end-start:.5f}\n')

# Note start time
start = time.time()
# Run function
test = BOPM_Fast(S, K, r, q, u, d, T, n, Option_Type)
# Note end time and calculate difference
end = time.time()
print("The value of our {0} option is {1:0.2f}".format(Option_Type,test))
print(f'The time elapsed whilst running the Fast Method with {n} time steps is {end-start:.5f}')

The value of our call option is 45.18
The time elapsed whilst running the Slow Method with 10 time steps is 0.00090

The value of our call option is 45.18
The time elapsed whilst running the Fast Method with 10 time steps is 0.00533


In [6]:
# Now we will use a large n and consider a put option
n = 5000
Option_Type = 'put'

# Note start time
start = time.time()
# Run function
BOPM_Slow(S, K, r, q, u, d, T, n, Option_Type)
# Note end time and calculate difference
end = time.time()
print(f'The time elapsed whilst running the Slow Method with {n} time steps is {end-start:.5f}\n')

# Note start time
start = time.time()
# Run function
test = BOPM_Fast(S, K, r, q, u, d, T, n, Option_Type)
# Note end time and calculate difference
end = time.time()
print("The value of our {0} option is {1:0.2f}".format(Option_Type,test))
print(f'The time elapsed whilst running the Fast Method with {n} time steps is {end-start:.5f}')

The value of our put option is 60.65
The time elapsed whilst running the Slow Method with 5000 time steps is 7.75591

The value of our put option is 60.65
The time elapsed whilst running the Fast Method with 5000 time steps is 0.00289


### Evidently, our fast implementation is far superior.

<h1> <ins>Model 2</ins> </h1>

From **"BOPM Derivations and Concepts"** (24-34) for a **call option**,
$$V = SB\left(a;n,\hat{p}\right) - Ke^{-rT}B\left(a;n,\bar{p}\right)$$\
$$\text{where}\,\,B\left(x;n,p\right) = \sum_{j=x}^{n}\binom{n}{j}p^{j}\left(1-p\right)^{n-j} = 1 - \mathbb{P}\left(X \le x-1\right)\,\,\text{where}\,\,X \thicksim \text{Bin}\left(n,p\right)$$\
$$\text{where}\,\,a \in \mathbb{N}\,\,\text{s.t}\,\,a > \frac{\ln{\left(\frac{K}{Sd^{n}}\right)}}{\ln{\left(\frac{u}{d}\right)}}$$\
$$\text{and}\,\,\bar{p} = \frac{e^{r\delta t}-d}{u-d}$$\
$$\text{and}\,\,\hat{p} = \frac{u}{e^{r\delta t}}\bar{p}$$\
$$\text{where}\,\,u = e^{\sigma \sqrt{\delta t}}\,\,\text{and}\,\,d = e^{-\sigma \sqrt{\delta t}}$$

And for a **put option**,
$$V = Ke^{-rT}\tilde{B}\left(a;n,\bar{p}\right) - S\tilde{B}\left(a;n,\hat{p}\right)$$\
$$\text{where}\,\,\tilde{B}\left(x;n,p\right) = \sum_{j=0}^{x}\binom{n}{j}p^{j}\left(1-p\right)^{n-j} = \mathbb{P}\left(X \le x\right)\,\,\text{where}\,\,X \thicksim \text{Bin}\left(n,p\right)$$\
$$\text{where}\,\,a \in \mathbb{N}\,\,\text{s.t}\,\,a < \frac{\ln{\left(\frac{K}{Sd^{n}}\right)}}{\ln{\left(\frac{u}{d}\right)}}$$\
$$\text{and}\,\,\bar{p} = \frac{e^{r\delta t}-d}{u-d}$$\
$$\text{and}\,\,\hat{p} = \frac{u}{e^{r\delta t}}\bar{p} = \frac{u\left(e^{r\delta t} - d\right)}{e^{r\delta t}\left(u-d\right)}$$\
$$\text{where}\,\,u = e^{\sigma \sqrt{\delta t}}\,\,\text{and}\,\,d = e^{-\sigma \sqrt{\delta t}}$$

Note how $ud = 1$. To avoid float division by zero errors, we will adjust the equations to calculate $a$.

So we will rewrite the definition of $a$ in the **call option** equation to
$$a \in \mathbb{N}\,\,\text{s.t}\,\,a > \frac{\ln{\left(\frac{K}{S}\right)} - n \times \ln{\left(d\right)}}{\ln{\left(\frac{u}{d}\right)}}$$

And similarly rewrite the definition of $a$ in the **put option** equation to
$$a \in \mathbb{N}\,\,\text{s.t}\,\,a < \frac{\ln{\left(\frac{K}{S}\right)} - n \times \ln{\left(d\right)}}{\ln{\left(\frac{u}{d}\right)}}$$

Now let us implement this.

In [7]:
def BOPM_CRR(S, K, r, q, sigma, T, n, Option_Type):

    # Check to see if we have a valid option type
    if Option_Type not in ('call','put'):
        print('Invalid option type, please enter either \'call\' or \'put\'')
        return
    
    # Calculate time step and discount factor
    dt = T/n
    df = np.exp(-r * dt)
    
    # Calculate our values for u and d
    u = np.exp(sigma * np.sqrt(dt))
    d = 1/u
    
    # Calculate our value for a
    ac = np.ceil((np.log(K/S) - (n * np.log(d))) / (np.log(u/d)))
    ap = np.floor((np.log(K/S) - (n * np.log(d))) / (np.log(u/d))) + 1
    
    # Calculate our 2 risk-neutral probabilities
    p1 = (np.exp((r-q) * dt) - d) / (u - d)
    p2 = u * df * p1
    
    # Create our B lambda function
    B = lambda x, n, p: binom(n, p).cdf(x-1)
    
    if Option_Type == 'call':
        # Calculate the inital value of the option if it is a call
        V = (S * (1 - B(ac, n, p2))) - (K * np.exp(-r * T) * (1 - B(ac, n, p1)))
        
    else:
        # Calculate the inital value of the option if it is a put
        V = (K * np.exp(-r * T) * B(ap, n, p1)) - (S * B(ap, n, p2))

    # Return the option price at the root
    #print("The value of our {0} option is {1:0.2f}".format(Option_Type,V))

    return V

In [12]:
# Test our function and time it

# We can use a large n and let us choose a value for volatility
n = 1000000
sigma = 0.2
Option_Type='call'

start = time.time()
test = BOPM_CRR(S, K, r, q, sigma, T, 1000000, Option_Type)
end = time.time()
print("The value of our {0} option is {1:0.2f}".format(Option_Type,test))
print(f'The time elapsed whilst running the CRR Model with {n} time steps is {end-start:.5f}')

The value of our call option is 41.62
The time elapsed whilst running the CRR Model with 1000000 time steps is 0.00276


<h1> <ins>Model 3</ins> </h1>

From **"BOPM Derivations and Concepts"** (24-34), all our equations are the same as above for our CRR model except how we define our parameters $u$ and $d$. From (33) and (34),

$$u = \dfrac{e^{r\delta t +\left(\sigma \sqrt{\delta t}\,\mathbin{/} \sqrt{\bar{p}\left(1-\bar{p}\right)}\right)}}{\bar{p}e^{\left(\sigma \sqrt{\delta t}\,\mathbin{/} \sqrt{\bar{p}\left(1-\bar{p}\right)}\right)} + \left(1-\bar{p}\right)}$$\
$$d = \dfrac{e^{r\delta t}}{\bar{p}e^{\left(\sigma \sqrt{\delta t}\,\mathbin{/} \sqrt{\bar{p}\left(1-\bar{p}\right)}\right)} + \left(1-\bar{p}\right)}$$
This is for some $\bar{p}\,\,$s.t $\,\,0<\bar{p}<1$.

Now let us implement this.

In [10]:
def BOPM_Chance(S, K, r, q, sigma, T, n, p, Option_Type):

    # Check to see if we have a valid option type
    if Option_Type not in ('call','put'):
        print('Invalid option type, please enter either \'call\' or \'put\'')
        return
    
    # Calculate time step and discount factor
    dt = T/n
    df = np.exp(-r * dt)
    
    # Calculate our values for u and d
    temp1 = np.exp((sigma*np.sqrt(dt))/(np.sqrt(p*(1-p))))
    u = (np.exp(r * dt)*temp1)/((p*temp1)+1-p)
    d = np.exp(r * dt)/((p*temp1)+1-p)
    
    # Calculate our value for a
    ac = np.ceil((np.log(K/S) - (n * np.log(d))) / (np.log(u/d)))
    ap = np.floor((np.log(K/S) - (n * np.log(d))) / (np.log(u/d))) + 1

    # Calculate our 2 risk-neutral probabilities
    p1 = (np.exp((r-q) * dt) - d) / (u - d)
    p2 = u * df * p1
    
    # Create our B lambda function
    B = lambda x, n, p: binom(n, p).cdf(x-1)
    
    if Option_Type == 'call':
        # Calculate the inital value of the option if it is a call
        V = (S * (1 - B(ac, n, p2))) - (K * np.exp(-r * T) * (1 - B(ac, n, p1)))
        
    else:
        # Calculate the inital value of the option if it is a put
        V = (K * np.exp(-r * T) * B(ap, n, p1)) - (S * B(ap, n, p2))
    
    # Return the option price at the root
    #print("The value of our {0} option is {1:0.2f}".format(Option_Type,V))
    
    return V

In [11]:
# Test our function and time it

# We can use a large n and let us choose a value for volatility
n = 1000000
sigma = 0.2
Option_Type='call'

start = time.time()
BOPM_Chance(S, K, r, q, sigma, T, 1000000, 0.5, Option_Type)
end = time.time()
print("The value of our {0} option is {1:0.2f}".format(Option_Type,test))
print(f'The time elapsed whilst running the Chance Model with {n} time steps is {end-start:.5f}')

The value of our call option is 41.62
The time elapsed whilst running the Chance Model with 1000000 time steps is 0.00647
